# Notebook 2 (V4_3) — Absorbing-Regime Balanced-Growth Path & Reduced State-by-State System

When the regime $z = b$ (absorbing), the equilibrium is characterised
by a stationary normalised solution. The V4_3 selection
(`ass_absorbing_stationary_equilibrium_selection`, line 947) further
imposes the common-growth condition $G_b = G_W$, which couples
$\nu_b$ to $\xi_W$ once the production blocks are calibrated.

With constant labour shares $\bar\varphi_b, \bar\varphi_W$ on the
absorbing branch (V4_3 Proposition `prop_balanced_growth_primitives`,
line 986):
$$
N_{US,t+1} = G_{N,US} N_{US,t}, \quad N_{W,t+1} = G_{N,W} N_{W,t},
$$
$$
Y_{US,t} \propto N_{US,t}^{\nu_b}, \quad
q_{US,t} \propto N_{US,t}^{\nu_b-1}, \quad
d_{US,t} \propto N_{US,t}^{\nu_b-1},
$$
and similarly for RoW with exponent $\xi_W$.

**Reduced 7-equation Markov competitive equilibrium system**
(V4_3 §"Reduced state-by-state characterization", lines 745–882).
Primary unknowns:
$$
\bar x_b = (\bar\varphi_{US}, \bar\varphi_W, \bar\omega,
            \bar\theta_{US}^*, \bar\omega^*, \bar R_f, \bar R_f^W).
$$
**Recovered** by zero-net-supply bond clearing (V4_3 lines 800–802):
$$
\bar\theta_W^* = 0,
\qquad
\bar\theta = -\bar\theta_{US}^* \frac{\bar A^*}{\bar A}.
$$

| F[i] | Equation | V4_3 label |
|---|---|---|
| F[1] | US stock-market value clearing | `intratemporal_phi_US` |
| F[2] | RoW stock-market value clearing | `intratemporal_phi_W` |
| F[3] | US $\omega$ FOC | `intratemporal_omega_US` |
| F[4] | US $\theta$ FOC (on recovered $\theta$) | `intratemporal_theta_US` |
| F[5] | RoW $\omega^*$ FOC | `intratemporal_omega_W` |
| F[6] | RoW $R_f^W = R_p^*$ in deterministic BGP | `intratemporal_thetaW` |
| F[7] | RoW $\theta_{US}^*$ clearing with $\chi/\theta_{US}^*$ wedge | `intratemporal_thetaUS` |


In [ ]:
using Pkg; Pkg.activate(".")
include("TwoCountryProductionOLG.jl")
using Plots, LaTeXStrings, Printf
gr()


In [ ]:
p = ProductionParams()   # HKT-matched defaults (see TwoCountryProductionOLG.jl §1)
validate_params(p)
println("Parameters validated.")


## 1. BGP at the Initial State $(N_{US,0}, N_{W,0})$

In [ ]:
bgp = solve_bgp_at(p, p.N_US_0, p.N_W_0)

println("═══ BGP at initial state ($(p.N_US_0), $(p.N_W_0)) ═══")
@printf("  φ_US = %.4f, φ_W = %.4f\n", bgp.φ_US, bgp.φ_W)
@printf("  ω = %.4f, ω* = %.4f\n", bgp.ω, bgp.ω_star)
@printf("  θ_US^* (primary, V4_3) = %+.4f\n", bgp.θ_US_star)
@printf("  θ      (recovered)     = %+.4f\n", bgp.θ)
@printf("  R_f = %.4f, R_f^W = %.4f\n", bgp.R_f, bgp.R_f_W)
@printf("  R_US = %.4f, R_W = %.4f, R_p = %.4f\n", bgp.R_US, bgp.R_W, bgp.R_p)
@printf("  R_A = %.4f, R_A^* = %.4f\n", bgp.R_A, bgp.R_A_star)
@printf("  G_N_US = %.4f, G_N_W = %.4f\n", bgp.G_N_US, bgp.G_N_W)
@printf("  G_b = %.4f, G_W = %.4f (gap = %+.2e)\n",
        bgp.G_N_US^p.ν_b, bgp.G_N_W^p.ξ_W,
        bgp.G_N_US^p.ν_b - bgp.G_N_W^p.ξ_W)
@printf("  Q_US = %.4f, Q_W = %.4f, e_US = %.4f, e_W = %.4f\n",
        bgp.Q_US, bgp.Q_W, bgp.e_US, bgp.e_W)
@printf("  I_US = %.4e, I_W = %.4e  (V4_3 country_ipo_transfer)\n",
        bgp.I_US, bgp.I_W)
@printf("  Mkt-cap identity gap (V4_3 Qsum_identity_mod_zerosupply): %+.2e\n",
        bgp.Q_US + bgp.Q_W - (p.β*bgp.e_US + (p.β+p.χ)/(1+p.χ)*bgp.e_W))
@printf("  Output identity gap (V4_3 country_output_income_identity), US: %+.2e\n",
        bgp.Y_US - (bgp.e_US + bgp.N_US*bgp.d_US - bgp.I_US))
@printf("  Ψ (V4_3 ass_regular_kernel) = %.4f\n", bgp.Psi)
@printf("  Converged: %s, ‖F‖ = %.2e\n", bgp.converged, bgp.residual_norm)


## 2. Common-World-Growth Calibration

V4_3 `ass_absorbing_stationary_equilibrium_selection` (line 947, common-growth
condition at line 980–983) requires $G_b = G_{N,US,b}^{\nu_b}$ to equal
$G_W = G_{N,W,b}^{\xi_W}$ — a maintained assumption of the bubble theorem
`thm_prod_sufficient`. **The benchmark enforces it by default**
(`common_world_growth=true`): `calibrate_common_growth` recalibrates $\nu_b$
(HKT seed $0.1 \to \approx 0.082$) by damped fixed-point iteration, holding all
other HKT-literal parameters fixed.

Why it matters: at HKT's literal $\nu_b=0.1$ the asymmetric portfolio block (US home
bias $\bar\omega=0.8$ vs RoW $\bar\omega^*=0.2$, plus the US-bond convenience yield
$\chi$) makes $\varphi_{US}^b \ne \varphi_W^b$, so $G_b \approx 1.00657$ exceeds
$G_W \approx 1.00532$ — a $\approx 0.13\%$/period gap that drifts relative country
size and would eventually break `ass_interior_equity_weights`. The `solve_bgp_at(p,…)`
cell above prints this gap at the seed $\nu_b$; the calibration below closes it to
machine zero.

In [ ]:
cal = calibrate_common_growth(p, p.N_US_0, p.N_W_0; verbose=false)
p_cal = cal.params
bgp_cal = cal.bgp

@printf("Calibrated ν_b = %.6f (was %.6f)\n", p_cal.ν_b, p.ν_b)
@printf("Common-growth gap (V4_3): G_b - G_W = %+.2e\n",
        bgp_cal.G_N_US^p_cal.ν_b - bgp_cal.G_N_W^p_cal.ξ_W)


## 3. BGP Stationarity Across Different $(N_{US}, N_W)$

In [ ]:
states = [(1.0, 1.0), (5.0, 5.0), (10.0, 10.0), (50.0, 50.0), (100.0, 100.0)]
println("BGP at multiple equal-N states (ν_b calibrated for common growth, p_cal):")
println("  N_US    N_W     φ_US    φ_W     ω      θ_US^*    θ        ω*     R_f    R_A     Ψ")
for (N_US, N_W) in states
    b = solve_bgp_at(p_cal, N_US, N_W)
    @printf("  %5.1f  %5.1f   %.4f  %.4f  %.4f  %+.4f  %+.4f  %.4f  %.4f  %.4f  %.4f\n",
            N_US, N_W, b.φ_US, b.φ_W, b.ω, b.θ_US_star, b.θ, b.ω_star,
            b.R_f, b.R_A, b.Psi)
end


## 4. Sensitivity to Relative Country Size $(N_W/N_{US})$

In [ ]:
ratios = [0.5, 1.0, 2.0, 5.0, 10.0]
results = [solve_bgp_at(p_cal, 1.0, r) for r in ratios]

p1 = plot(ratios, [r.φ_US for r in results], lw=2, marker=:circle, label=L"\bar\varphi_{US}",
          xlabel=L"N_W/N_{US}", ylabel=L"\bar\varphi", xscale=:log10,
          title="BGP labour allocations")
plot!(p1, ratios, [r.φ_W for r in results], lw=2, marker=:square, label=L"\bar\varphi_W")

p2 = plot(ratios, [r.ω for r in results], lw=2, marker=:circle, label=L"\bar\omega",
          xlabel=L"N_W/N_{US}", ylabel="weight", xscale=:log10,
          title="BGP portfolio weights")
plot!(p2, ratios, [r.ω_star for r in results], lw=2, marker=:square, label=L"\bar\omega^*")

# V4_3 primary bond unknown θ_US^* (positive); θ recovered (negative).
p3 = plot(ratios, [r.θ_US_star for r in results], lw=2, marker=:circle,
          label=L"\bar\theta_{US}^*  \mathrm{(primary, > 0)}",
          xlabel=L"N_W/N_{US}", ylabel="bond share", xscale=:log10,
          title="BGP bond shares (V4_3: θ_US^* primary; θ recovered)")
plot!(p3, ratios, [r.θ for r in results], lw=2, marker=:square,
      label=L"\bar\theta  \mathrm{(recovered, < 0)}")

p4 = plot(ratios, [r.R_f for r in results], lw=2, marker=:circle, label=L"R_f",
          xlabel=L"N_W/N_{US}", ylabel="return", xscale=:log10,
          title="BGP risk-free rates")
plot!(p4, ratios, [r.R_f_W for r in results], lw=2, marker=:square, label=L"R_f^W")
plot!(p4, ratios, [r.R_A for r in results], lw=2, marker=:utriangle, label=L"R_A")

plot(p1, p2, p3, p4, layout=(2,2), size=(900, 700))


## 5. Per-Variety Prices and Aggregate Market Cap along BGP

V4_3 Proposition `prop_balanced_growth_primitives` (line 986, eqs.
`bg_us_per_variety_scaling`, `bg_us_scaling`):
$$
q_{US,t} = \bar q_{US} N_{US,t}^{\nu_b-1},
\qquad
\mathcal Q_{US,t} = \bar{\mathcal Q}_{US} N_{US,t}^{\nu_b}.
$$


In [ ]:
Ns = 10 .^ range(0, 4, length=120)
qs = Float64[]; ds = Float64[]; Qs = Float64[]; Rs = Float64[]
for N in Ns
    b = solve_bgp_at(p_cal, N, N)
    push!(qs, b.q_US); push!(ds, b.d_US); push!(Qs, b.Q_US); push!(Rs, b.R_US)
end

p1 = plot(Ns, qs, lw=2, label=L"q_{US}^b", xscale=:log10, yscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="per-variety price (log)",
          title="Per-variety stock price along BGP",
          titlefontsize=10, legend=:bottomleft, legendfontsize=9)
plot!(p1, Ns, ds, lw=2, label=L"d_{US}^b", ls=:dash)

p2 = plot(Ns, Qs, lw=2, label=L"\mathcal{Q}_{US}^b", xscale=:log10, yscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="aggregate market cap (log)",
          title="Aggregate stock-market capitalisation",
          titlefontsize=10, legend=:topleft, legendfontsize=9)

p3 = plot(Ns, Rs, lw=2, label=L"R_{US}^b", xscale=:log10,
          xlabel=L"N_{US}=N_W", ylabel="return",
          title="Per-variety return on absorbing BGP",
          titlefontsize=10, legend=:topright, legendfontsize=9)
hline!(p3, [1.0], ls=:dash, color=:black, label="")

plot(p1, p2, p3, layout=(1,3), size=(1200, 400),
     left_margin=5Plots.mm, bottom_margin=5Plots.mm, top_margin=3Plots.mm)


## 6. Verifying the Aggregate Market-Cap Identity (V4_3 eq. `Qsum_identity_mod_zerosupply`)

In [ ]:
println("V4_3 Aggregate market-cap identity (eq. Qsum_identity_mod_zerosupply):")
println("   N_US    Q_US+Q_W    βe_US+(β+χ)/(1+χ)e_W    gap         Y-identity gap (US)")
for N in [1.0, 5.0, 10.0, 50.0, 100.0]
    b = solve_bgp_at(p_cal, N, N)
    LHS = b.Q_US + b.Q_W
    RHS = p.β*b.e_US + (p.β+p.χ)/(1+p.χ)*b.e_W
    y_gap = b.Y_US - (b.e_US + b.N_US*b.d_US - b.I_US)
    @printf("  %5.1f   %.4f       %.4f                %+.2e   %+.2e\n",
            N, LHS, RHS, LHS-RHS, y_gap)
end


## Summary

- The reduced 7-equation Markov system (V4_3 lines 818–855) is solved
  in primary unknowns $(\bar\varphi_{US}, \bar\varphi_W, \bar\omega,
  \bar\theta_{US}^*, \bar\omega^*, \bar R_f, \bar R_f^W)$, with bond
  shares $\bar\theta$ and $\bar\theta_W^*$ recovered from zero-net-supply
  clearing.
- Common-world-growth calibration adjusts $\nu_b$ via damped fixed-point
  iteration so that $G_b = G_W$
  (V4_3 `ass_absorbing_stationary_equilibrium_selection`).
- The V4_3 aggregate market-cap identity
  $\mathcal Q_{US}+\mathcal Q_W = \beta e_{US} + \tfrac{\beta+\chi}{1+\chi}e_W$
  (eq. `Qsum_identity_mod_zerosupply`) holds exactly at every BGP point.
- The V4_3 output identity
  $Y_i = e_i + \mathcal D_i - \mathcal I_i$
  (eq. `country_output_income_identity`) is satisfied to machine
  precision at every BGP.
- The regularity wedge $\Psi$ (V4_3 `ass_regular_kernel`) is reported
  in every BGP solution and should be $> 0$ for Theorem 1.

Next notebook: solve the **unbalanced branch** by forward-backward
iteration over the reduced 7-equation Markov system.
